# 03 | Desemprego (BLS)

Esta parte prepara o indicador econômico de desemprego. Não precisamos de gráficos difíceis: basta ter um valor correto por estado e ano.

**Como usar:** execute os blocos de código 1, 2 e 3, nessa ordem. Leia a explicação antes de cada bloco. Cada bloco reúne pequenas ações que fazem parte da mesma etapa.

## Bloco 1: Ler a planilha

Antes, extraia `staadata.xlsx` do ZIP do BLS para `data/raw/staadata/`.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import pandas as pd
arquivo = ROOT / "data/raw/staadata/staadata.xlsx"
bls = pd.read_excel(arquivo, sheet_name=0, header=None, skiprows=8, usecols=[0,1,2,9], names=["fips","state","year","unemployment_rate"])
print("Linhas lidas:",len(bls))

## Bloco 2: Selecionar os estados e anos

Usamos os mesmos 50 estados do CDC para que os nomes combinem quando juntarmos os arquivos.

In [ ]:
cdc = pd.read_csv(ROOT / "data/raw/cdc_homicides_2019_2024.csv")
estados = set(cdc["name"]) - {"District of Columbia"}
bls["year"] = pd.to_numeric(bls["year"], errors="coerce")
limpo = bls[bls["state"].isin(estados) & bls["year"].between(2019,2024)][["state","year","unemployment_rate"]].copy()
limpo["year"] = limpo["year"].astype(int)
print("Dimensões:",limpo.shape)
display(limpo.head())

## Bloco 3: Validar e salvar

Uma taxa deve estar entre 0% e 100%, e cada estado precisa aparecer uma vez em cada ano.

In [ ]:
assert limpo.shape == (300,3)
assert not limpo.duplicated(["state","year"]).any()
assert limpo["unemployment_rate"].notna().all()
assert limpo["unemployment_rate"].between(0,100).all()
destino = ROOT / "data/processed/bls_unemployment_2019_2024.csv"
destino.parent.mkdir(parents=True,exist_ok=True)
limpo.to_csv(destino,index=False)
print("OK, salvo:",destino.relative_to(ROOT))